# TPI – Introducción al Análisis de Datos (División 343)
**Caso de estudio:** Telco Customer Churn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rociojuarez/analisis-de-datos-telco-churn/blob/main/notebooks/TPI_Analisis_de_Datos.ipynb)


---
# PARTE 1 – Fundamentos, ingesta y limpieza de datos

## Paso 1 · Problema de negocio, hipótesis y variable objetivo

**¿Qué es esto?** Antes de tocar datos hay que decir *para qué* los vamos a analizar.

- **Problema de negocio:** una empresa de telecomunicaciones pierde clientes que cancelan su servicio (*churn*). Retener a un cliente cuesta menos que conseguir uno nuevo, así que queremos entender **qué características tienen los clientes que se van** para poder actuar antes.
- **Variable objetivo (target):** `Churn` (Yes/No). Es lo que queremos explicar o predecir.
- **Hipótesis analíticas:** afirmaciones que vamos a verificar con los datos más adelante (en el EDA):
  1. Los clientes con contrato *Month-to-month* cancelan más que los de contratos largos.
  2. Los clientes con poca antigüedad (`tenure` baja) cancelan más.
  3. Cargos mensuales altos (`MonthlyCharges`) se asocian a más cancelación.
  4. Pagar con *Electronic check* se asocia a más cancelación.

> ✏️ **Tu turno:** podés cambiar o agregar hipótesis, pero tienen que poder comprobarse con columnas del dataset.

## Paso 2 · Cargar los datos y primer vistazo

**¿Qué hacemos?** Leemos el CSV con `pandas` en un *DataFrame* (una tabla en memoria) y miramos su tamaño, sus primeras filas y el tipo de dato (`dtype`) que pandas le asignó a cada columna.

In [ ]:
import pandas as pd
import numpy as np

# Ruta al dato original (data/raw). En Colab se va a leer desde el repo de GitHub (lo configuramos al final).
RUTA_RAW = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(RUTA_RAW)
print("Filas, columnas:", df.shape)
df.head()

In [ ]:
# Tipo de dato que pandas asignó por defecto a cada columna
df.dtypes

### Lectura rápida de lo que acabamos de ver
- Hay **7043 filas** (clientes) y **21 columnas** (variables).
- `TotalCharges` aparece como `object` (texto), aunque debería ser un número. **Esa es la primera trampa del dataset** y la resolvemos en el Paso 4.
- `SeniorCitizen` aparece como `int64` (0/1), pero en realidad es una categoría (Sí/No).

## Paso 3 · Clasificar el dataset y las variables

**Tipo de dato según su estructura:** el dataset es **estructurado**: está organizado en filas (clientes) y columnas (variables) con un formato fijo, como una tabla de base de datos. (Semi-estructurado sería algo como JSON; no estructurado, texto libre, imágenes o audio.)

**Tipos de variable.** Primero lo que importa:
- **Cualitativa (categórica):** describe una cualidad. **Nominal** = categorías sin orden (ej. género). **Ordinal** = categorías con orden natural (ej. contrato corto < largo).
- **Cuantitativa (numérica):** es una cantidad. **Discreta** = valores contables (ej. meses). **Continua** = puede tomar cualquier valor en un rango (ej. dinero).

| Variable | Tipo teórico | `dtype` de pandas | ¿Coincide? |
|---|---|---|---|
| `customerID` | Cualitativa nominal (identificador, no sirve para analizar) | object | ✅ |
| `gender`, `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling` | Cualitativa nominal (binaria) | object | ✅ |
| `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `PaymentMethod` | Cualitativa nominal | object | ✅ |
| `SeniorCitizen` | Cualitativa nominal (binaria) | **int64** | ❌ es categoría guardada como número |
| `Contract` | Cualitativa **ordinal** (Month-to-month < One year < Two year) | object | ✅ (pandas no sabe que hay orden) |
| `tenure` | Cuantitativa **discreta** (meses de antigüedad) | int64 | ✅ |
| `MonthlyCharges` | Cuantitativa **continua** | float64 | ✅ |
| `TotalCharges` | Cuantitativa **continua** | **object** | ❌ es número guardado como texto |
| `Churn` | Cualitativa nominal (binaria) – **variable objetivo** | object | ✅ |

> 💡 **Para la defensa oral:** sabé explicar por qué `SeniorCitizen` y `TotalCharges` no coinciden. Es exactamente lo que pide el enunciado (“contrastar la clasificación teórica con los dtypes de pandas”).